# Spectral signatures from Sentinel-2: forest, fields, water and snow

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/spectral_signatures_en.ipynb)

**OpenSkyLab · notebook for the visual** [Spectral signatures](https://openskylab.space/edu_viz_spectral_signatures.html) · [česká verze](spectral_signatures_cs.ipynb)

The visual shows how chlorophyll, leaf water, dry matter and LAI shape the spectral curve of vegetation and why NDVI stops growing in dense canopies. Here you check it on real Sentinel-2 pixels near Roztoky and on a time series of one field over the 2024 season.

**Goal:** compare real spectral curves of surfaces, locate the red edge, show NDVI saturation on a field time series, and explain why the same index from different bands gives a different number.

**Data:** Sentinel-2 L2A from the Earth Search catalogue on AWS: scene `S2B_33UVR_20240721_0_L2A` (21 July 2024, near Roztoky), cloud-free scenes of tile 33UVR from March to October 2024 for the time series, and `S2A_33UWS_20240308_0_L2A` (snow in the Krkonoše mountains); ESA WorldCover 2021 classes. About 30 MB is downloaded; the time series takes 1 to 3 minutes.

**Tasks:** six. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 45 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

## 1. Setup

In [ ]:
%pip install -q rasterio

In [ ]:
import requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from pyproj import Transformer

STAC = "https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/"
WORLDCOVER = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N48E012_Map.tif"
BANDS = ["coastal", "blue", "green", "red", "rededge1", "rededge2", "rededge3", "nir", "nir08", "swir16", "swir22"]

def read_on_grid(url, grid, resampling=Resampling.bilinear, dtype=np.float32):
    """Read a cloud raster and resample it to grid = (transform, crs, height, width)."""
    tr, crs, ny, nx = grid
    out = np.zeros((ny, nx), dtype)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY=5, GDAL_HTTP_RETRY_DELAY=2):
        with rasterio.open("/vsicurl/" + url) as src:
            reproject(rasterio.band(src, 1), out, dst_transform=tr, dst_crs=crs, resampling=resampling)
    return out

def s2_window(item_id, lon, lat, half):
    """Reflectance of all Sentinel-2 L2A bands in a square around a point (10 m grid in the tile's UTM)."""
    item = requests.get(STAC + item_id, timeout=60).json()
    epsg = item["properties"]["proj:epsg"] if "proj:epsg" in item["properties"] else int(item["properties"]["proj:code"].split(":")[1])
    x, y = Transformer.from_crs(4326, epsg, always_xy=True).transform(lon, lat)
    n = int(2 * half / 10)
    grid = (rasterio.transform.from_origin(x - half, y + half, 10, 10), f"EPSG:{epsg}", n, n)
    # since processing baseline 04.00 (2022) DN carry a +1000 offset; Earth Search has already removed it for newer items
    p = item["properties"]
    offset = -1000 if float(p.get("s2:processing_baseline", "0")) >= 4 and not p.get("earthsearch:boa_offset_applied", False) else 0
    refl = {b: (read_on_grid(item["assets"][b]["href"], grid) + offset) / 10000 for b in BANDS}
    scl = read_on_grid(item["assets"]["scl"]["href"], grid, Resampling.nearest, np.uint8)
    wl = {b: item["assets"][b]["eo:bands"][0]["center_wavelength"] for b in BANDS}
    return refl, scl, wl, grid, item

## 2. Spectral curves of real surfaces

In [ ]:
refl, scl, wl, grid, item = s2_window("S2B_33UVR_20240721_0_L2A", 14.375, 50.145, 2000)
wc = read_on_grid(WORLDCOVER, grid, Resampling.nearest, np.uint8)
clear = np.isin(scl, [4, 5, 6])
classes = {10: "forest", 30: "meadow", 40: "field", 50: "built-up", 80: "water"}
lam = np.array([wl[b] for b in BANDS])
fig, ax = plt.subplots(figsize=(10, 4))
curves = {}
for code, name in classes.items():
    m = clear & (wc == code)
    stack = np.array([refl[b][m] for b in BANDS])
    q25, q50, q75 = np.percentile(stack, [25, 50, 75], axis=1)
    curves[name] = q50
    ax.plot(lam, q50, "o-", label=name); ax.fill_between(lam, q25, q75, alpha=0.15)
ax.set(xlabel="wavelength (µm)", ylabel="reflectance", title="Median and interquartile range, Sentinel-2 L2A, 21 July 2024")
ax.legend(); plt.show()

> **✏️ Task 1.** Describe the forest curve: why is it low in blue and red, with a small peak in green and a steep rise between 0.70 and 0.78 µm? Why does the field (in late July often ripe grain or stubble) have more red and less NIR than the forest? Why is the spread so large for built-up areas and fields?

*Your answer:* 

## 3. Red edge position

The red edge is the steep rise in reflectance between red and NIR. We estimate its position (REP) by linear interpolation from bands B4, B5, B6 and B7 (Guyot and Baret 1988). More chlorophyll shifts the edge to longer wavelengths.

In [ ]:
def rep(r4, r5, r6, r7):
    mid = (r4 + r7) / 2                                   # reflectance halfway up the edge
    return 705 + 35 * (mid - r5) / (r6 - r5)             # nm (B5 = 705 nm, B6 = 740 nm)

REP = rep(refl["red"], refl["rededge1"], refl["rededge2"], refl["rededge3"])
for code, name in classes.items():
    m = clear & (wc == code)
    print(f"{name:9s}: REP median {np.nanmedian(REP[m]):.1f} nm")

> **✏️ Task 2.** Which surface has the red edge furthest and why? Does it make sense to compute REP for water or built-up areas? Look at the values and explain why the index has no physical meaning without vegetation even though it returns a number.

*Your answer:* 

## 4. NDVI saturation during the season

We take one field pixel near Únětice and all cloud-free images from 2024. We compare NDVI = (NIR − R)/(NIR + R), NDRE = (NIR − RE1)/(NIR + RE1) and NDMI = (NIR − SWIR1)/(NIR + SWIR1).

In [ ]:
lon, lat = 14.3516, 50.1632          # ◀ parameter: point (field near Únětice); also try forest, e.g. 14.366, 50.148
search = requests.post("https://earth-search.aws.element84.com/v1/search", json={
    "collections": ["sentinel-2-l2a"], "intersects": {"type": "Point", "coordinates": [lon, lat]},
    "datetime": "2024-03-01T00:00:00Z/2024-10-31T23:59:59Z", "limit": 100,
    "query": {"eo:cloud_cover": {"lt": 40}, "grid:code": {"eq": "MGRS-33UVR"}}}, timeout=60).json()
print("scenes in the catalogue:", len(search["features"]))

def pixel(item, key, x, y):
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY=5, GDAL_HTTP_RETRY_DELAY=2):
        with rasterio.open("/vsicurl/" + item["assets"][key]["href"]) as r:
            row, col = r.index(x, y)
            return float(r.read(1, window=Window(col - 1, row - 1, 3, 3)).mean() if key != "scl"
                         else r.read(1, window=Window(col, row, 1, 1))[0, 0])

rows = []
for it in sorted(search["features"], key=lambda f: f["properties"]["datetime"]):
    x, y = Transformer.from_crs(4326, 32633, always_xy=True).transform(lon, lat)
    if pixel(it, "scl", x, y) not in (4, 5):               # only clear vegetation or bare soil
        continue
    p = it["properties"]
    off = -1000 if float(p.get("s2:processing_baseline", "0")) >= 4 and not p.get("earthsearch:boa_offset_applied", False) else 0
    r = {k: (pixel(it, k, x, y) + off) / 10000 for k in ("red", "rededge1", "nir08", "nir", "swir16")}
    r["date"] = it["properties"]["datetime"][:10]
    rows.append(r)
ts = pd.DataFrame(rows).drop_duplicates("date").set_index("date")
ts["NDVI"] = (ts.nir - ts.red) / (ts.nir + ts.red)
ts["NDRE"] = (ts.nir - ts.rededge1) / (ts.nir + ts.rededge1)
ts["NDMI"] = (ts.nir08 - ts.swir16) / (ts.nir08 + ts.swir16)
ax = ts[["NDVI", "NDRE", "NDMI"]].plot(marker="o", figsize=(10, 3.8), rot=45)
ax.set(ylabel="index", title=f"Point {lon}, {lat}: {len(ts)} clear scenes in 2024"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(ts.NDRE, ts.NDVI)
for d, r in ts.iterrows(): ax.annotate(d[5:], (r.NDRE, r.NDVI), fontsize=7)
ax.set(xlabel="NDRE", ylabel="NDVI", title="Same date, two indices"); plt.show()

> **✏️ Task 3.** When did the field reach its peak and when was it harvested? In what NDVI range does the NDVI–NDRE plot bend, i.e. NDVI hardly grows but NDRE still does? Explain the saturation: why can red not drop any lower at high LAI while the red edge still moves?

*Your answer:* 

> **✏️ Task 4.** How does NDMI behave during the season and why does it drop earlier or faster than NDVI as the crop ripens? Change the point to forest (`14.366, 50.148`) and compare: why is the forest series much flatter?

*Your answer:* 

## 5. Water and snow

NDWI (green and NIR, McFeeters 1996) highlights water, NDSI (green and SWIR 1.6 µm) highlights snow. Snow is bright in the visible but absorbs in the SWIR; clouds reflect in both.

In [ ]:
snow, snow_scl, *_ = s2_window("S2A_33UWS_20240308_0_L2A", 15.70, 50.735, 3000)
ndsi = (snow["green"] - snow["swir16"]) / (snow["green"] + snow["swir16"])
ndwi = (refl["green"] - refl["nir"]) / (refl["green"] + refl["nir"])
thr = 0.4                               # ◀ parameter: NDSI threshold for snow (MODIS uses 0.4)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
rgb = np.dstack([snow["red"], snow["green"], snow["blue"]]); ax[0].imshow(np.clip(rgb / 1.0, 0, 1)); ax[0].set_title("Krkonoše 8 March 2024, true colour")
ax[1].imshow(ndsi > thr, cmap="Blues"); ax[1].set_title(f"NDSI > {thr}: {np.mean(ndsi > thr):.0%} of the area")
im = ax[2].imshow(ndwi, cmap="BrBG", vmin=-0.8, vmax=0.8); ax[2].set_title("NDWI near Roztoky (Vltava)"); fig.colorbar(im, ax=ax[2])
for a in ax: a.axis("off")
plt.show()
print(f"agreement with the SCL snow mask: {np.mean((ndsi > thr) == (snow_scl == 11)):.1%} of pixels")

> **✏️ Task 5.** Why does NDSI separate snow from clouds although both are white in the visible? Change the threshold to 0.2 and 0.6: where does the mask change most (snowy forest, shadows, dwarf pine)? Why is the Vltava positive in NDWI and vegetation negative?

*Your answer:* 

## 6. Same index, different number

NDVI can be computed from Sentinel-2 with the broad band B8 (842 nm, 145 nm wide, 10 m) or the narrow B8A (865 nm, 33 nm, 20 m). A drone multispectral camera or Landsat have different bands again. We compare both variants on the same pixels.

In [ ]:
ndvi_b8 = (refl["nir"] - refl["red"]) / (refl["nir"] + refl["red"])
ndvi_b8a = (refl["nir08"] - refl["red"]) / (refl["nir08"] + refl["red"])
m = clear & np.isin(wc, [10, 30, 40])
d = (ndvi_b8a - ndvi_b8)[m]
print(f"NDVI(B8A) − NDVI(B8) in vegetation: mean {d.mean():+.3f}, 5th–95th percentile {np.percentile(d, 5):+.3f} to {np.percentile(d, 95):+.3f}")
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.hexbin(ndvi_b8[m], ndvi_b8a[m], gridsize=60, bins="log", cmap="viridis"); ax.axline((0, 0), slope=1, color="r", lw=1)
ax.set(xlabel="NDVI with B8 (10 m)", ylabel="NDVI with B8A (20 m)"); plt.show()

> **✏️ Task 6.** How much do the two NDVIs differ and in which direction? Suggest two causes (band position and width, 10 vs 20 m resolution). What follows when you want to compare drone NDVI with Sentinel-2 NDVI, or old and new NDVI maps?

*Your answer:* 

---
### Data sources and references

- Copernicus Sentinel-2 data 2024, ESA (L2A, Sen2Cor); COG copy in the AWS Open Data registry (`sentinel-cogs`), Earth Search catalogue (Element 84). Free use with attribution under the [Legal notice on the use of Copernicus Sentinel Data](https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice).
- Zanaga, D. et al. (2022). *ESA WorldCover 10 m 2021 v200*. https://doi.org/10.5281/zenodo.7254221 (CC BY 4.0).
- Red edge position by linear interpolation: Guyot, G., Baret, F. (1988). Utilisation de la haute résolution spectrale pour suivre l'état des couverts végétaux. *Proc. 4th Int. Colloquium on Spectral Signatures of Objects in Remote Sensing*, ESA SP-287, 279–286.
- NDVI saturation and red-edge indices: Gitelson, A. A., Merzlyak, M. N. (1994). Spectral reflectance changes associated with autumn senescence of *Aesculus hippocastanum* L. and *Acer platanoides* L. leaves. *Journal of Plant Physiology* 143, 286–292. https://doi.org/10.1016/S0176-1617(11)81633-0
- NDWI: McFeeters, S. K. (1996). The use of the Normalized Difference Water Index (NDWI) in the delineation of open water features. *International Journal of Remote Sensing* 17, 1425–1432. https://doi.org/10.1080/01431169608948714
- NDSI for snow: Hall, D. K., Riggs, G. A., Salomonson, V. V. (1995). Development of methods for mapping global snow cover using MODIS. *Remote Sensing of Environment* 54, 127–140. https://doi.org/10.1016/0034-4257(95)00137-P

### How to cite

OpenSkyLab (2026). Spectral signatures from Sentinel-2: forest, fields, water and snow [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/spectral_signatures_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*